# Exploración visual de Gold

Este notebook consume únicamente los Parquet generados por el pipeline. No contiene datos ni credenciales y permite visualizar los resultados durante la sustentación.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
GOLD_ROOT = PROJECT_ROOT / "out" / "gold"
GOLD_ROOT

In [ ]:
def read_gold(dataset: str) -> pd.DataFrame:
    path = GOLD_ROOT / dataset / "current" / "part-00000.parquet"
    if not path.exists():
        raise FileNotFoundError(
            f"No existe {path}. Ejecute primero el pipeline con destino local."
        )
    return pd.read_parquet(path)

pairs = read_gold("product_pairs")
recommendations = read_gold("cross_sell_recommendations")
customer_360 = read_gold("customer_360")

{
    "product_pairs": len(pairs),
    "recommendations": len(recommendations),
    "customers_360": len(customer_360),
}

## Pares con mayor afinidad explicable

Se filtran pares con evidencia mínima y se muestran los de mayor lift. Un lift alto sin suficiente frecuencia puede ser inestable, por eso se conserva también `invoices_together`.

In [ ]:
top_pairs = (
    pairs.loc[pairs["invoices_together"] >= 2]
    .nlargest(15, "lift")
    .assign(pair=lambda frame: frame["product_a"].astype(str) + " → " + frame["product_b"].astype(str))
    .sort_values("lift")
)

ax = top_pairs.plot.barh(
    x="pair",
    y="lift",
    figsize=(10, 7),
    legend=False,
    color="#2563eb",
)
ax.set(title="Top 15 pares de productos por lift", xlabel="Lift", ylabel="Par de productos")
ax.axvline(1.0, color="black", linestyle="--", linewidth=1)
plt.tight_layout()
plt.show()

## Distribución de oportunidades por cliente

In [ ]:
counts = recommendations.groupby("customer_id").size()
ax = counts.value_counts().sort_index().plot.bar(figsize=(9, 5), color="#16a34a")
ax.set(
    title="Cantidad de recomendaciones vigentes por cliente",
    xlabel="Recomendaciones",
    ylabel="Clientes",
)
plt.tight_layout()
plt.show()